# Full-image coverage: fresh YOLO and ConvNeXt U-Net on T4

Run the **single cell below** in a Colab T4 GPU runtime. This is a new experiment; old solar model weights do not initialize it. YOLO starts from generic COCO YOLO26s-seg weights. U-Net starts with an ImageNet ConvNeXt-Tiny encoder and a new decoder.

- Each epoch uses **616 GONG + 250 annotated SWEFil + 616 existing CHASE synthetic images**. Every 2048 image contributes its four nonoverlapping 1024 tiles: **5,928 tiles per model per epoch**, including empty tiles. Shuffle changes order, not coverage. There is no 400-crop cap or domain-probability sampling.
- The two models run **sequentially, 100 epochs each** on the same T4. `MODELS` selects their order or just one model. `RUN_UNTIL_EPOCH` limits this invocation for each selected model while retaining its 100-epoch optimizer schedule. Already completed work is reused.
- Existing CHASE synthesis and the pinned annotated SWEFil download cache are reused. GONG validation images remain outside GONG training. All 250 annotated SWEFil observations are included as requested; nearby external observations mean the validation score remains exploratory.
- Run the **same cell** after interruption or runtime reset. Saved source is verified and reused from Drive without fetching GitHub. Model state is verified through the Drive API. Use the same account for Drive mounting and authentication.
- `PHASE="prepare"` prepares data without GPU training. Progress is relayed to Colab and appended to `OUTPUT/training.log`. The review ZIP contains metrics and metadata; model weights stay in their verified checkpoint storage.

This notebook has CPU fixture coverage for bootstrap and resume behavior. Training runs on your Colab GPU; its validation or leaderboard outcome is not known in advance.


In [ ]:
# One cell for first launch and resume after interruption or runtime reset.
PHASE = "all"                 # "prepare" downloads/restores data without training.
MODELS = ["yolo", "unet"]     # Sequentially; use one name to run only that model.
RUN_UNTIL_EPOCH = 100          # Stage cap; the optimizer schedule always remains 100.
from pathlib import Path
from google.colab import auth, drive, files
import hashlib, json, os, re, signal, subprocess, sys
BASE = Path("/content/drive/MyDrive/filament-segmentation-2026")
OUTPUT = BASE / "fixed_split_experiments/fullpass_both_t4_v1"
CHASE_TRAINING_ROOT = BASE / "fixed_split_experiments/chase_shape_best84_recipe_t4_v1"
CACHE_ROOT = Path("/content/fullpass_both_t4_cache")
REPOSITORY = "https://github.com/ikjoo5/solar_filament.git"
CHECKOUT = Path("/content/solar_fullpass_both_github")
drive.mount("/content/drive")
# Drive API verification is required before a checkpoint is committed.
auth.authenticate_user()
OUTPUT = OUTPUT.resolve()
def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
state_path = OUTPUT / "bootstrap_state.json"
if state_path.is_file():
    state = json.loads(state_path.read_text())
    if state.get("scope") != "fullpass_both_t4_v1" or state.get("output_root") != str(OUTPUT):
        raise ValueError("Saved bootstrap belongs to another experiment.")
    version = state.get("version", "")
    if not re.fullmatch("[0-9a-f]{16}", version):
        raise ValueError("Invalid saved source version.")
    snapshot = OUTPUT / "code" / version
    manifest_path = snapshot / "source_manifest.json"
    if manifest_path.is_symlink() or not snapshot.resolve().is_relative_to(OUTPUT) or sha(manifest_path) != state["manifest_sha256"]:
        raise ValueError("Saved source manifest changed; existing files were preserved.")
    manifest = json.loads(manifest_path.read_text())
    identity = json.dumps(manifest, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    if hashlib.sha256(identity).hexdigest()[:16] != version:
        raise ValueError("Saved source version differs from its manifest.")
    for name, expected in manifest.items():
        path = snapshot / name
        if path.is_symlink() or not path.resolve().is_relative_to(snapshot.resolve()):
            raise ValueError("Unsafe saved source path.")
        if not path.is_file() or sha(path) != expected:
            raise ValueError("Saved launcher/source changed: " + name)
    bootstrap = snapshot / "tools/fullpass_both_colab.py"
    if "tools/fullpass_both_colab.py" not in manifest:
        raise ValueError("Saved manifest does not contain the launcher.")
else:
    if CHECKOUT.exists():
        origin = subprocess.check_output(["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True).strip()
        if origin != REPOSITORY or dirty:
            raise ValueError("Existing checkout differs or has local edits. Preserve it and choose another CHECKOUT.")
        subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(CHECKOUT)], check=True)
    bootstrap = CHECKOUT / "tools/fullpass_both_colab.py"
    if not bootstrap.is_file():
        raise FileNotFoundError("The repository does not contain the full-pass launcher.")
command = [sys.executable, "-u", str(bootstrap), "--base", str(BASE),
           "--output", str(OUTPUT), "--cache-root", str(CACHE_ROOT),
           "--chase-training-root", str(CHASE_TRAINING_ROOT),
           "--phase", PHASE, "--run-until", str(RUN_UNTIL_EPOCH),
           "--models", ",".join(MODELS)]
process = subprocess.Popen(command, start_new_session=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, encoding="utf-8", errors="replace", bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)
finally:
    if process.poll() is None:
        try: os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError: pass
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try: os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError: pass
            process.wait()
    process.stdout.close()
result = json.loads((OUTPUT / "latest_result.json").read_text())
print("Status:", result.get("status"), "Completed epochs by model:", result.get("epochs_completed", {}))
if result.get("archive") and result.get("status") in ("complete", "completed", "paused"):
    archive = Path(result["archive"]).resolve()
    if not archive.is_relative_to(OUTPUT) or not archive.is_file() or sha(archive) != result.get("archive_sha256"):
        raise ValueError("Result archive is missing or changed; no download attempted.")
    print("Review ZIP:", archive)
    files.download(str(archive))
else:
    print("Preparation results:", OUTPUT / "latest_result.json")
